# Agents 
智能体将语言模型与工具相结合，创建能够对任务进行推理、决定使用哪些工具并迭代地寻求解决方案的系统。
> create_agent 使用 LangGraph 构建基于图的代理运行时。图由节点（步骤）和边（连接）组成，定义了代理如何处理信息。代理在这个图中移动，执行诸如模型节点（调用模型）、工具节点（执行工具）或中间件之类的节点。

## Core components  核心组件
### Model  模型
#### Static model  静态模型
静态模型在创建代理时配置一次，并在整个执行过程中保持不变。这是最常见、最直接的方法。



In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

base_url = os.getenv("GLM_BASE_URL")
api_key = os.getenv("GLM_API_KEY")

basic_model = ChatOpenAI(
    model="glm-4.5-air",
    base_url=base_url,
    api_key=api_key
)

#### Dynamic model  动态模型
动态模型的选择在runtime根据当前情况state 以及 context。这支持了复杂的路由逻辑和成本优化。
要使用动态模型，使用修改请求中模型的 @wrap_model_call 装饰器创建中间件：

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.agents import create_agent

advence_model = ChatOpenAI(
    model="glm-4.5",
    base_url=base_url,
    api_key=api_key
)

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advence_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    # tools=tools,
    middleware=[dynamic_model_selection]
)

> 使用结构化输出时不支持预装模型（bind_tools 已调用的模型）。如果你需要动态选择模型并带结构化输出，确保传递给中间件的模型没有预绑。

### Tools  工具
工具赋予代理采取行动的能力。代理不仅仅限模型工具绑定，还促进了：
- 多个工具调用依次触发（由一个提示触发）
- 适当时并行工具调用
- 基于以往结果的动态工具选择
- 工具重试逻辑与错误处理
- 工具调用间的状态持久性

#### Static tools  静态工具
静态工具在创建代理时定义，执行过程中保持不变。

工具可以被指定为普通的 Python 函数，或者coroutines  协程.

工具装饰器可用于自定义工具名称、描述、参数模式及其他属性。

In [ ]:
from langchain.tools import tool


# @tool
# def search(query: str) -> str:
#     """Search for information."""
#     return f"Results for: {query}"

@tool
def search(query: str) -> str:
    """Search for information."""
    raise RuntimeError("Simulated tool failure")

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

tools = [search, get_weather]

#### Dynamic tools  动态工具
动态工具则是在运行时修改代理可用的工具集，而不是事先定义。并非所有工具都适合所有情况。过多工具可能会使模型不堪重负（上下文过载），并增加错误;限制能力太少。动态工具选择允许根据认证状态、用户权限、功能标志或对话阶段调整可用工具集。
##### Filtering pre-registered tools 过滤预注册工具
当所有可能的工具在代理创建时都已知，你可以预先注册它们，并根据状态、权限或上下文动态过滤哪些工具暴露在模型中。

###### state
只有在达到特定对话里程碑后才启用高级工具：

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on conversation State."""
    # Read from State: check if user has authenticated
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])

    # Only enable sensitive tools after authentication
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    elif message_count < 5:
        # Limit tools early in conversation
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model=basic_model,
    tools=[public_search, private_search, advanced_search],
    middleware=[state_based_tools]
)

###### store
基于用户偏好或商店功能标志的过滤工具：

In [ ]:
@wrap_model_call
def store_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on Store preferences."""
    user_id = request.runtime.context.user_id

    # Read from Store: get user's enabled features
    store = request.runtime.store
    feature_flags = store.get(("features",), user_id)

    if feature_flags:
        enabled_features = feature_flags.value.get("enabled_tools", [])
        # Only include tools that are enabled for this user
        tools = [t for t in request.tools if t.name in enabled_features]
        request = request.override(tools=tools)

    return handler(request)

##### Runtime Context 运行时上下文
基于运行时上下文用户权限的过滤工具：

In [ ]:
@wrap_model_call
def context_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on Runtime Context permissions."""
    # Read from Runtime Context: get user role
    if request.runtime is None or request.runtime.context is None:
        # If no context provided, default to viewer (most restrictive)
        user_role = "viewer"
    else:
        user_role = request.runtime.context.user_role

    if user_role == "admin":
        # Admins get all tools
        pass
    elif user_role == "editor":
        # Editors can't delete
        tools = [t for t in request.tools if t.name != "delete_data"]
        request = request.override(tools=tools)
    else:
        # Viewers get read-only tools
        tools = [t for t in request.tools if t.name.startswith("read_")]
        request = request.override(tools=tools)

    return handler(request)

- 所有可能的工具在编译/启动时都已知
- 需要根据权限、功能标志或对话状态进行筛选
- 工具是静态的，但它们的可用性是动态的

#### Runtime tool registration 运行时工具注册
当工具在运行时被发现或创建（例如从 MCP 服务器加载、基于用户数据生成，或从远程注册表获取），你需要既注册工具，还需要动态处理其执行。

这需要两个中间件钩子：
1. `wrap_model_call` - 将动态工具添加到请求中
2. `wrap_tool_call` - 处理动态添加工具的执行

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ToolCallRequest

# A tool that will be added dynamically at runtime
@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 20.0) -> str:
    """Calculate the tip amount for a bill."""
    tip = bill_amount * (tip_percentage / 100)
    return f"Tip: ${tip:.2f}, Total: ${bill_amount + tip:.2f}"

class DynamicToolMiddleware(AgentMiddleware):
    """Middleware that registers and handles dynamic tools."""

    def wrap_model_call(self, request: ModelRequest, handler):
        # Add dynamic tool to the request
        # This could be loaded from an MCP server, database, etc.
        updated = request.override(tools=[*request.tools, calculate_tip])
        return handler(updated)

    def wrap_tool_call(self, request: ToolCallRequest, handler):
        # Handle execution of the dynamic tool
        if request.tool_call["name"] == "calculate_tip":
            return handler(request.override(tool=calculate_tip))
        return handler(request)

agent = create_agent(
    model=basic_model,
    tools=[],  # Only static tools registered here
    middleware=[DynamicToolMiddleware()],
)

# The agent can now use both get_weather AND calculate_tip
result = agent.invoke({
    "messages": [{"role": "user", "content": "Calculate a 20% tip on $85"}]
})
result

- 工具是在运行时发现的（例如，来自 MCP 服务器）
- 工具是基于用户数据或配置动态生成的
- 正在与外部工具注册库集成
> wrap_tool_call 钩子对于运行时注册工具是必需的，因为代理需要知道如何执行原始工具列表中没有的工具。没有它，代理就无法调用这个动态添加的工具。

#### Tool error handling  工具错误处理
要自定义工具错误的处理方式，可以使用 @wrap_tool_call 装饰器创建中间件：

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )
agent = create_agent(
    model=basic_model,
    tools=tools,
    middleware=[handle_tool_errors]
)
res = agent.invoke({
    "messages": [{"role": "user", "content": "Search for: good"}]
})
print(res)


#### Tool use in the ReAct loop ReAct 循环中的工具使用
代理遵循 ReAct（“推理+行动”）模式，交替进行简短的推理步骤和针对性工具调用，并将所得观察反馈到后续决策中，直到能够给出最终答案。

### System prompt  系统提示
可以通过提供提示来塑造你的代理人处理任务的方式。system_prompt 参数可以作为字符串表示：

In [ ]:
system_prompt="You are a helpful assistant. Be concise and accurate."

当没有提供 system_prompt 时，代理会直接从消息推断出任务。

system_prompt 参数接受 str 或 SystemMessage。使用 SystemMessage 可以让你更好地控制提示词结构

In [ ]:
from langchain.messages import SystemMessage

system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You are an AI assistant tasked with analyzing literary works.",
            },
            {
                "type": "text",
                "text": "<the entire contents of 'Pride and Prejudice'>",
                "cache_control": {"type": "ephemeral"}
            }
        ]
    )

cache_control 字段中带有 {“type”： “ephemeral”} 的字段指示 Anthropic 缓存该内容块，从而降低使用同一系统提示符重复请求时的延迟和成本。


### Dynamic system prompt  动态系统提示
对于需要根据运行时上下文或代理状态修改系统提示符的高级用例，可以使用中间件 。
`@dynamic_prompt` 装饰器创建中间件，根据模型请求生成系统提示：

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model=basic_model,
    # tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)
result

### Name  名称
为代理人设定一个可选名字 。在多智能体系统中，添加代理作为子图时，这作为节点标识符：
```python
agent = create_agent(
    model,
    tools,
    name="research_assistant"
)
```
一些模型提供者会拒绝包含空格或带有错误特殊字符的名称。使用字母数字字符、下划线和连字符，仅能确保所有提供者之间的兼容性。工具名称也是一样。

## Advanced concepts  高级概念
### Structured output  结构化输出
LangChain 通过 `response_format` 参数提供结构化输出的策略。
#### ToolStrategy  工具策略
`ToolStrategy` 使用人工工具调用来生成结构化输出。这适用于任何支持工具调用的模型。当提供者原生结构化输出（通过 `ProviderStrategy`）不可用或不可靠时，应使用 `ToolStrategy`。

In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=basic_model,
    # tools=[search_tool],
    # response_format=ToolStrategy(ContactInfo)
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

#### ProviderStrategy
`ProviderStrategy` 使用模型提供者的原生结构化输出生成。这种方法更可靠，但只适用于支持原生结构化输出的提供者

从 langchain 1.0 起，仅传递一个模式（例如 response_format=ContactInfo）如果模型支持原生结构化输出，则默认使用 ProviderStrategy。否则它会退回 ToolStrategy。

### Memory  记忆
代理通过消息状态自动维护对话历史。你还可以配置代理使用自定义状态模式，以便在对话中记住额外信息。

存储在状态中的信息可以看作是代理的短期记忆 ：

自定义状态模式必须将 AgentState 扩展为 TypedDict。

定义自定义状态有两种方式：
1. 通过中间件 （优先）
2. 通过 state_schema 在 create_agent
#### Defining state via middleware 通过中间件定义状态
当需要通过特定中间件钩子和附加在中间件上的工具访问自定义状态时，使用中间件来定义自定义状态。


In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any


class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    # tools = [tool1, tool2]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        ...

agent = create_agent(
    model,
    tools=tools,
    middleware=[CustomMiddleware()]
)

# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

#### Defining state via state_schema 通过 state_schema 定义状态
使用 state_schema 参数作为快捷方式定义仅在工具中使用的自定义状态。
> 从 langchain 1.0 起，自定义状态模式必须是 TypedDict 类型。
```python
class CustomState(AgentState):
    user_preferences: dict

agent = create_agent(
    model,
    tools=[tool1, tool2],
    state_schema=CustomState
)
```
> 通过中间件定义自定义状态比通过 `state_schema on create_agent `更受青睐，因为这样可以让你在概念上保持状态扩展对应相关中间件和工具的范围。

### Streaming  流媒体

In [ ]:
from langchain.messages import AIMessage, HumanMessage

agent = create_agent(
    model=basic_model,
    # tools=tools,
    # middleware=[CustomMiddleware()]
)

for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

### Middleware  中间件
中间件为在不同执行阶段定制代理行为提供了强大的扩展性。你可以利用中间件来：
- 调用模型前的进程状态（例如，消息修剪、上下文注入）
- 修改或验证模型的响应（例如，护栏、内容过滤）
- 用自定义逻辑处理工具执行错误
- 基于状态或上下文实现动态模型选择
- 添加自定义日志、监控或分析功能